# Tiny Dreamer Highway — Colab Baseline Real Run

**Name:** Esteban  
**Course:** CSC 580 AI 2  
**Assignment:** Final Project — Dream the Road  
**AI tools consulted:** GitHub Copilot

Use this notebook for the first real baseline run before any non-trivial optimization changes.

## Baseline intent

This notebook should be your comparison point for later optimization work. Keep the seed, runtime type, and run length stable when comparing results.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
REPO_URL = 'https://github.com/estmon8u/CSC_580_Final_Project.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/CSC_580_Final_Project')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

for path in [DRIVE_ROOT, ARTIFACT_ROOT, ARTIFACT_ROOT / 'training_runs']:
    path.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%%bash
set -e
REPO_URL='https://github.com/estmon8u/CSC_580_Final_Project.git'
if [ ! -d /content/CSC_580_Final_Project/.git ]; then
  git clone "${REPO_URL}" /content/CSC_580_Final_Project
else
  cd /content/CSC_580_Final_Project
  git pull --ff-only origin main
fi
cd /content/CSC_580_Final_Project
python -m pip install --upgrade pip --quiet
python -m pip install -e . --quiet

Updating bf2d7f2..3dd2169
Fast-forward
 README.md                                          |   2 +
 docs/colab_git_drive_workflow.md                   |  11 +
 docs/configuration_reference.md                    |   4 +-
 examples/h100_amp_experiment.yaml                  |   6 +-
 examples/h100_experiment.yaml                      |   4 +-
 examples/h100_screening_experiment.yaml            |   6 +-
 examples/optimized_experiment.yaml                 |   6 +-
 examples/training_run.yaml                         |   6 +-
 notebooks/02_colab_sanity_run.ipynb                | 405 +++------------------
 notebooks/03_colab_baseline_run.ipynb              | 178 +--------
 notebooks/04_colab_h100_run.ipynb                  |  27 +-
 notebooks/05_colab_optimized_run.ipynb             |  33 +-
 notebooks/06_colab_h100_amp_run.ipynb              |  29 +-
 notebooks/07_colab_screening_run.ipynb             |  21 +-
 notebooks/README.md                                |   2 +
 src/tiny_dreamer_highw

From https://github.com/estmon8u/CSC_580_Final_Project
 * branch            main       -> FETCH_HEAD
   bf2d7f2..3dd2169  main       -> origin/main


In [ ]:
import json
import sys
import torch
from pathlib import Path

PROJECT_ROOT = Path('/content/CSC_580_Final_Project')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tiny_dreamer_highway.config import load_experiment_config
from tiny_dreamer_highway.training import run_training_experiment

CONFIG_PATH = PROJECT_ROOT / 'examples' / 'training_run.yaml'
config = load_experiment_config(CONFIG_PATH)

def print_sequence_sampling_guidance(run_label, warm_start_steps):
    effective_warm_start = config.training.warm_start_steps if warm_start_steps is None else warm_start_steps
    nominal_floor = config.training.batch_size * config.replay.sequence_length
    print(f'{run_label} sequence length:', config.replay.sequence_length)
    print(f'{run_label} max episode steps:', config.env.max_episode_steps)
    print(f'{run_label} offroad terminal:', config.env.reward.offroad_terminal)
    print(f'{run_label} effective warm-start steps:', effective_warm_start)
    print(f'{run_label} nominal warm-start floor (batch x sequence):', nominal_floor)
    print('Trainer safeguard: extra random warm-start data will be collected automatically if valid sequences are still unavailable.')
    if config.replay.sequence_length > config.env.max_episode_steps:
        print('Warning: sequence_length exceeds max_episode_steps, so replay sequence sampling is impossible.')
    elif config.env.reward.offroad_terminal and config.replay.sequence_length >= config.env.max_episode_steps - 8:
        print('Caution: sequence_length is close to the episode horizon while offroad_terminal=True; short crash-heavy warm starts may require extra random collection.')
    elif effective_warm_start < nominal_floor:
        print('Caution: warm-start is below the nominal batch_size x sequence_length floor.')

print('Loaded config from:', CONFIG_PATH)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('Batch size:', config.training.batch_size)
print('World-model updates/cycle:', config.training.world_model_updates_per_cycle)
print('Behavior updates/cycle:', config.training.behavior_updates_per_cycle)

Loaded config from: /content/CSC_580_Final_Project/examples/training_run.yaml
GPU: NVIDIA A100-SXM4-80GB
Batch size: 32
World-model updates/cycle: 2
Behavior updates/cycle: 2


In [ ]:
RUN_NAME = 'baseline_real_run_001'
RUN_ARTIFACT_ROOT = ARTIFACT_ROOT / 'training_runs' / RUN_NAME
RUN_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

CYCLES = None
WARM_START_STEPS = None
POLICY_STEPS = None
CHECKPOINT_INTERVAL = None
RESUME_FROM = None

print('Run name:', RUN_NAME)
print('Effective cycles:', config.training.cycles if CYCLES is None else CYCLES)
print('Effective policy steps:', config.training.policy_steps if POLICY_STEPS is None else POLICY_STEPS)
print_sequence_sampling_guidance('Baseline run', WARM_START_STEPS)

Run name: baseline_real_run_001
Effective cycles: 500


In [ ]:
print('Launching baseline real run. Per-cycle progress lines will appear below.')

training_summary = run_training_experiment(
    config,
    RUN_ARTIFACT_ROOT,
    cycles=CYCLES,
    warm_start_steps=WARM_START_STEPS,
    policy_steps=POLICY_STEPS,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    resume_from=RESUME_FROM,
)

print('Completed cycles:', training_summary.completed_cycles)
print('Latest checkpoint:', training_summary.latest_checkpoint)
print('Latest metrics:', training_summary.latest_record)

Launching baseline real run. Per-cycle progress lines will appear below.
[train] starting run | cycles=500 | start_step=1 | warm_start_steps=1500 | policy_steps=32 | device=cuda
[train] step=1/500 | warm=1500 | policy=32 | replay=1532 | world_total=5655.4482 | actor=-0.1963 | critic=2.1484 | eval_reward=n/a | cycle_s=637.0 | elapsed_s=637.0 | checkpoint=-
[train] step=2/500 | warm=0 | policy=32 | replay=1564 | world_total=5240.6033 | actor=-0.2510 | critic=2.0368 | eval_reward=n/a | cycle_s=14.1 | elapsed_s=651.1 | checkpoint=-
[train] step=3/500 | warm=0 | policy=32 | replay=1596 | world_total=4843.3186 | actor=-0.1658 | critic=2.1027 | eval_reward=n/a | cycle_s=14.2 | elapsed_s=665.3 | checkpoint=-
[train] step=4/500 | warm=0 | policy=32 | replay=1628 | world_total=4615.4917 | actor=0.0259 | critic=2.0919 | eval_reward=n/a | cycle_s=14.4 | elapsed_s=679.7 | checkpoint=-
[train] step=5/500 | warm=0 | policy=32 | replay=1660 | world_total=4465.8574 | actor=0.0110 | critic=2.1273 | eval

In [ ]:
summary_path = training_summary.log_dir / 'latest_summary.json'
summary_payload = json.loads(summary_path.read_text(encoding='utf-8'))
summary_payload

In [ ]:
from IPython.display import Image, display
import csv
import importlib
import json
import tiny_dreamer_highway.evaluation.training_analysis as training_analysis

training_analysis = importlib.reload(training_analysis)
export_training_history_artifacts = training_analysis.export_training_history_artifacts

analysis_outputs = export_training_history_artifacts(
    training_summary.log_dir / 'cycle_metrics.csv',
    RUN_ARTIFACT_ROOT / 'analysis',
    prefix=RUN_NAME,
)

display(Image(filename=str(analysis_outputs['curves'])))
analysis_summary = json.loads(analysis_outputs['summary'].read_text(encoding='utf-8'))
with (training_summary.log_dir / 'cycle_metrics.csv').open('r', encoding='utf-8', newline='') as handle:
    metric_rows = list(csv.DictReader(handle))
latest_metrics_row = metric_rows[-1]
{
    'analysis_summary': analysis_summary,
    'phase4_metrics': {
        'world_model/reconstruction_loss': latest_metrics_row.get('world_model/reconstruction_loss'),
        'world_model/reconstruction_mse': latest_metrics_row.get('world_model/reconstruction_mse'),
        'world_model/observation_log_prob': latest_metrics_row.get('world_model/observation_log_prob'),
        'world_model/overshooting_kl_loss': latest_metrics_row.get('world_model/overshooting_kl_loss'),
        'world_model/overshooting_feature_mse': latest_metrics_row.get('world_model/overshooting_feature_mse'),
        'evaluation/mean_reward': latest_metrics_row.get('evaluation/mean_reward'),
        'evaluation/crash_rate': latest_metrics_row.get('evaluation/crash_rate'),
    },
}

## Agent Driving Demo

Record agent-driving GIFs using the trained policy from this run. These show the actor driving in the real highway-env.

In [ ]:
from IPython.display import Image, display
import importlib
import tiny_dreamer_highway.evaluation as evaluation_pkg

try:
    evaluation_pkg = importlib.reload(evaluation_pkg)
    record_demo_videos = evaluation_pkg.record_demo_videos
except (AttributeError, ImportError):
    from tiny_dreamer_highway.evaluation.policy_rollout import record_demo_videos

print('Using demo recorder from:', record_demo_videos.__module__)

demo_outputs = record_demo_videos(
    config,
    checkpoint_path=training_summary.latest_checkpoint,
    output_dir=RUN_ARTIFACT_ROOT / 'demo_videos',
    num_episodes=3,
    max_steps=1000,
    fps=15,
    seed=config.seed,
    prefix=RUN_NAME,
    device=config.device,
)

for gif_path in demo_outputs.video_paths:
    print(f'\n{gif_path.name}')
    display(Image(filename=str(gif_path)))